# 08. XGBoost & LightGBM

## Learning Objectives
- Understand the Gradient Boosting concept
- XGBoost usage and hyperparameters
- LightGBM features and optimization
- CatBoost overview
- Model comparison and selection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.datasets import make_classification, load_breast_cancer, fetch_california_housing
import time

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 1. Gradient Boosting Concept

In [ ]:
print("""
Gradient Boosting Algorithm:

1. Initialize: F_0(x) = argmin_gamma sum L(y_i, gamma)

2. Iterate (m = 1, 2, ..., M):
   a. Compute pseudo-residuals:
      r_im = -[dL(y_i, F(x_i))/dF(x_i)]_{F=F_{m-1}}
   
   b. Train weak learner h_m(x) on the residuals
   
   c. Compute optimal step size:
      gamma_m = argmin_gamma sum L(y_i, F_{m-1}(x_i) + gamma * h_m(x_i))
   
   d. Update model:
      F_m(x) = F_{m-1}(x) + learning_rate * gamma_m * h_m(x)

Key ideas:
- Each step learns from the errors (residuals) of the previous model
- Optimizes in the gradient direction of the loss function
- learning_rate prevents overfitting
""")

# Simple visualization data
np.random.seed(42)
X_demo = np.linspace(0, 10, 100).reshape(-1, 1)
y_demo = np.sin(X_demo).ravel() + np.random.randn(100) * 0.3

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(X_demo, y_demo, alpha=0.5)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Sample Data for Gradient Boosting')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
stages = [0, 1, 5, 20, 50]
colors = ['red', 'orange', 'yellow', 'green', 'blue']
for stage, color in zip(stages, colors):
    if stage == 0:
        plt.axhline(y=np.mean(y_demo), color=color, label=f'Stage {stage}', alpha=0.7)
plt.scatter(X_demo, y_demo, alpha=0.3, color='gray')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Gradient Boosting: Sequential Learning')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. XGBoost (eXtreme Gradient Boosting)

In [ ]:
# XGBoost installation: pip install xgboost
import xgboost as xgb
from xgboost import XGBClassifier, XGBRegressor

print("""
XGBoost Features:

1. Regularization:
   - L1 and L2 regularization to prevent overfitting
   - Objective: sum L(y_i, y_hat_i) + sum Omega(f_k)
   - Omega(f) = gamma*T + 0.5*lambda*||w||^2

2. Efficient computation:
   - Uses 2nd-order Taylor expansion
   - Histogram-based splitting
   - Cache optimization

3. Missing value handling:
   - Automatically learns optimal direction

4. Parallel processing:
   - Parallel split point search per feature
""")

print(f"XGBoost version: {xgb.__version__}")

### 2.1 XGBoost Classification

In [ ]:
# Load Breast Cancer data
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Data shape: {X.shape}")
print(f"Classes: {cancer.target_names}")

In [ ]:
# Why: XGBoost adds L1/L2 regularization (reg_alpha, reg_lambda) to the loss,
# which plain GBM lacks. subsample and colsample_bytree introduce randomness
# (like RF) to reduce overfitting. These defaults (no sampling, lambda=1) are
# conservative — tune them on validation data for better generalization.
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    min_child_weight=1,     # Minimum weight for leaf node
    gamma=0,                # Minimum loss reduction for split
    subsample=1.0,          # Row sampling ratio
    colsample_bytree=1.0,   # Column sampling ratio per tree
    reg_alpha=0,            # L1 regularization
    reg_lambda=1,           # L2 regularization
    random_state=42,
    eval_metric='logloss'
)

# Training
start_time = time.time()
xgb_clf.fit(X_train, y_train)
train_time = time.time() - start_time

# Prediction and evaluation
y_pred = xgb_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=== XGBoost Classification Results ===")
print(f"Train accuracy: {xgb_clf.score(X_train, y_train):.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Training time: {train_time:.4f} sec")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

### 2.2 Early Stopping

In [ ]:
# Split validation data
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Why: Early stopping monitors validation loss and stops when it hasn't improved
# for N rounds. This automatically finds the optimal n_estimators, avoiding both
# underfitting (too few trees) and overfitting (too many trees).
xgb_early = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    early_stopping_rounds=10,  # Stop if no improvement for 10 rounds
    eval_metric='logloss'
)

xgb_early.fit(
    X_train_sub, y_train_sub,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("=== Early Stopping Results ===")
print(f"Best iteration: {xgb_early.best_iteration}")
print(f"Best score: {xgb_early.best_score:.4f}")
print(f"Test accuracy: {xgb_early.score(X_test, y_test):.4f}")

### 2.3 Feature Importance

In [ ]:
# Feature importance visualization
importance_types = ['weight', 'gain', 'cover']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, imp_type in zip(axes, importance_types):
    importance_dict = xgb_clf.get_booster().get_score(importance_type=imp_type)
    
    if importance_dict:
        # Show only top 10
        sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:10]
        features = [x[0] for x in sorted_importance]
        values = [x[1] for x in sorted_importance]
        
        ax.barh(range(len(features)), values)
        ax.set_yticks(range(len(features)))
        ax.set_yticklabels(features)
        ax.set_xlabel('Importance')
        ax.set_title(f'Feature Importance ({imp_type})')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("""
Importance types:
- weight: Number of times a feature is used in splits
- gain: Average gain when the feature is used
- cover: Average number of samples covered by the feature
""")

### 2.4 XGBoost Hyperparameter Tuning

In [ ]:
# Parameter grid
param_grid_xgb = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 200],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Grid Search (using simplified grid to save time)
grid_search_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid_xgb,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

# Use sample to save time
X_sample, _, y_sample, _ = train_test_split(X_train, y_train, train_size=0.3, random_state=42)
grid_search_xgb.fit(X_sample, y_sample)

print("=== XGBoost Grid Search Results ===")
print(f"Best parameters: {grid_search_xgb.best_params_}")
print(f"Best CV score: {grid_search_xgb.best_score_:.4f}")
print(f"Test score: {grid_search_xgb.score(X_test, y_test):.4f}")

## 3. LightGBM

In [ ]:
# LightGBM installation: pip install lightgbm
import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor

print("""
LightGBM Features:

1. Leaf-wise growth:
   - Traditional: Level-wise (horizontal split)
   - LightGBM: Leaf-wise (split leaf with maximum loss reduction)
   - Faster and more accurate but higher overfitting risk

2. Histogram-based splitting:
   - Discretizes continuous values
   - Memory efficient, fast training

3. GOSS (Gradient-based One-Side Sampling):
   - Samples based on gradient magnitude

4. EFB (Exclusive Feature Bundling):
   - Bundles mutually exclusive features
   - Effective for sparse features
""")

print(f"LightGBM version: {lgb.__version__}")

### 3.1 LightGBM Classification

In [ ]:
# Why: LightGBM grows leaf-wise (best-first) instead of level-wise, making it
# faster and often more accurate. num_leaves directly controls model complexity —
# it's more important than max_depth in LightGBM. Rule of thumb: num_leaves < 2^max_depth.
lgb_clf = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,           # -1: no limit
    num_leaves=31,          # Maximum number of leaf nodes
    min_child_samples=20,   # Minimum samples in leaf node
    subsample=1.0,          # Row sampling
    colsample_bytree=1.0,   # Column sampling
    reg_alpha=0,            # L1 regularization
    reg_lambda=0,           # L2 regularization
    random_state=42,
    verbose=-1
)

# Training
start_time = time.time()
lgb_clf.fit(X_train, y_train)
train_time_lgb = time.time() - start_time

# Evaluation
y_pred_lgb = lgb_clf.predict(X_test)
accuracy_lgb = accuracy_score(y_test, y_pred_lgb)

print("=== LightGBM Classification Results ===")
print(f"Train accuracy: {lgb_clf.score(X_train, y_train):.4f}")
print(f"Test accuracy: {accuracy_lgb:.4f}")
print(f"Training time: {train_time_lgb:.4f} sec")

### 3.2 num_leaves vs max_depth

In [ ]:
print("""
Relationship between num_leaves and max_depth:
- When max_depth = d, maximum leaf count = 2^d
- num_leaves = 31 is roughly equivalent to max_depth = 5
- To prevent overfitting: num_leaves < 2^max_depth

Recommended settings:
- Large data: num_leaves = 2^max_depth - 1 or less
- Small data: keep num_leaves small (15~31)
""")

# Performance by num_leaves
num_leaves_range = [15, 31, 63, 127, 255]
train_scores_lgb = []
test_scores_lgb = []

for num_leaves in num_leaves_range:
    lgb_temp = LGBMClassifier(
        n_estimators=100,
        num_leaves=num_leaves,
        random_state=42,
        verbose=-1
    )
    lgb_temp.fit(X_train, y_train)
    train_scores_lgb.append(lgb_temp.score(X_train, y_train))
    test_scores_lgb.append(lgb_temp.score(X_test, y_test))

plt.figure(figsize=(10, 6))
plt.plot(num_leaves_range, train_scores_lgb, 'o-', label='Train')
plt.plot(num_leaves_range, test_scores_lgb, 's-', label='Test')
plt.xlabel('num_leaves')
plt.ylabel('Accuracy')
plt.title('LightGBM: num_leaves Effect')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 3.3 Feature Importance

In [ ]:
# LightGBM feature importance
importance_lgb = pd.DataFrame({
    'Feature': cancer.feature_names,
    'Importance': lgb_clf.feature_importances_
}).sort_values('Importance', ascending=True).tail(15)

plt.figure(figsize=(10, 8))
plt.barh(importance_lgb['Feature'], importance_lgb['Importance'])
plt.xlabel('Importance')
plt.title('LightGBM Feature Importance - Top 15')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. CatBoost Overview

In [ ]:
print("""
CatBoost Features:

1. Automatic categorical feature handling:
   - Automatic Target Encoding
   - Prevents data leakage via Ordered Target Statistics

2. Ordered Boosting:
   - Randomizes training order to reduce bias
   - Prevents overfitting

3. Symmetric trees:
   - All nodes at the same level use the same split condition
   - Faster prediction speed

Installation: pip install catboost

Basic usage:
from catboost import CatBoostClassifier

cat_clf = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    random_state=42,
    verbose=False
)

cat_clf.fit(X_train, y_train)
""")

## 5. Boosting Algorithm Comparison

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Why: Comparing sklearn GBM, XGBoost, and LightGBM on the same data reveals
# speed vs accuracy tradeoffs. sklearn GBM is single-threaded; XGBoost parallelizes
# split finding; LightGBM uses histogram binning for the fastest training.
models = {
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
}

# Comparison
print("Boosting Algorithm Comparison:")
print("-" * 70)
print(f"{'Model':<20} {'Train Accuracy':>15} {'Test Accuracy':>15} {'Time (sec)':>15}")
print("-" * 70)

results = {}
for name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    results[name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'time': train_time
    }
    
    print(f"{name:<20} {train_acc:>15.4f} {test_acc:>15.4f} {train_time:>15.4f}")

print("-" * 70)

In [ ]:
# Visualization comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
names = list(results.keys())
test_accuracies = [results[n]['test_accuracy'] for n in names]
axes[0].barh(names, test_accuracies, color=['skyblue', 'salmon', 'lightgreen'])
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xlim([0.9, 1.0])
axes[0].grid(True, alpha=0.3)

# Training time comparison
times = [results[n]['time'] for n in names]
axes[1].barh(names, times, color=['skyblue', 'salmon', 'lightgreen'])
axes[1].set_xlabel('Training Time (seconds)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Regression Example

In [ ]:
# California Housing data
housing = fetch_california_housing()
X_reg, y_reg = housing.data, housing.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f"Data shape: {X_reg.shape}")
print(f"Features: {housing.feature_names}")
print(f"Target: Median house value (in $100,000s)")

In [ ]:
# XGBoost regression
xgb_reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)
xgb_reg.fit(X_train_reg, y_train_reg)
y_pred_xgb_reg = xgb_reg.predict(X_test_reg)

# LightGBM regression
lgb_reg = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
lgb_reg.fit(X_train_reg, y_train_reg)
y_pred_lgb_reg = lgb_reg.predict(X_test_reg)

# Evaluate
print("=== XGBoost Regression ===")
print(f"R2 Score: {r2_score(y_test_reg, y_pred_xgb_reg):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_xgb_reg)):.4f}")

print("\n=== LightGBM Regression ===")
print(f"R2 Score: {r2_score(y_test_reg, y_pred_lgb_reg):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_lgb_reg)):.4f}")

In [ ]:
# Prediction vs Actual visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost
axes[0].scatter(y_test_reg, y_pred_xgb_reg, alpha=0.5)
axes[0].plot([y_test_reg.min(), y_test_reg.max()], 
             [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'XGBoost (R2={r2_score(y_test_reg, y_pred_xgb_reg):.4f})')
axes[0].grid(True, alpha=0.3)

# LightGBM
axes[1].scatter(y_test_reg, y_pred_lgb_reg, alpha=0.5, color='green')
axes[1].plot([y_test_reg.min(), y_test_reg.max()], 
             [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'LightGBM (R2={r2_score(y_test_reg, y_pred_lgb_reg):.4f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Hyperparameter Guide

In [ ]:
# Hyperparameter comparison table
params_comparison = pd.DataFrame({
    'Parameter': ['Learning rate', 'Number of trees', 'Depth', 'Number of leaves', 'L1 regularization', 'L2 regularization', 'Row sampling', 'Column sampling'],
    'XGBoost': ['learning_rate', 'n_estimators', 'max_depth', '-', 'reg_alpha', 'reg_lambda', 'subsample', 'colsample_bytree'],
    'LightGBM': ['learning_rate', 'n_estimators', 'max_depth', 'num_leaves', 'reg_alpha', 'reg_lambda', 'subsample', 'colsample_bytree'],
    'Effect': ['Lower = more stable', 'More = more accurate', 'Deeper = more complex', 'More = more complex', 'Prevents overfitting', 'Prevents overfitting', 'Reduces variance', 'Increases diversity']
})

print("Hyperparameter Guide:")
print(params_comparison.to_string(index=False))

In [ ]:
print("""
Recommended tuning order:

1. Tree structure parameters:
   - max_depth, num_leaves
   - min_child_weight, min_child_samples

2. Sampling parameters:
   - subsample
   - colsample_bytree

3. Regularization parameters:
   - reg_alpha, reg_lambda

4. Learning rate adjustment:
   - Lower learning_rate
   - Increase n_estimators

Overfitting prevention strategies:
- Early stopping (early_stopping_rounds)
- Regularization (reg_alpha, reg_lambda)
- Sampling (subsample, colsample_bytree)
- Tree constraints (max_depth, min_child_weight)
- Lower learning rate (learning_rate)
""")

## Summary

### Algorithm Comparison

| Algorithm | Features | Pros | Cons |
|-----------|----------|------|------|
| Gradient Boosting | Residual learning | High accuracy | Slow training |
| XGBoost | Regularization + parallelization | Fast, accurate | Memory usage |
| LightGBM | Leaf-wise | Very fast, large-scale | Overfitting risk |
| CatBoost | Categorical handling | Less tuning needed | Slow start |

### Selection Guide

- **Small data (<10K)**: XGBoost or sklearn GradientBoosting
- **Medium data (10K-100K)**: XGBoost
- **Large data (>100K)**: LightGBM
- **Many categorical features**: CatBoost
- **Fast training needed**: LightGBM
- **Best accuracy**: Try all, then ensemble

### Key Hyperparameters

**Common:**
- `n_estimators`: Number of trees
- `learning_rate`: Learning rate
- `max_depth`: Tree depth

**XGBoost specific:**
- `min_child_weight`: Minimum leaf node weight
- `gamma`: Minimum loss reduction for split

**LightGBM specific:**
- `num_leaves`: Maximum number of leaf nodes
- `min_child_samples`: Minimum samples in leaf node

### Next Steps
- Stacking and Blending
- AutoML (Optuna, Hyperopt)
- Hands-on Kaggle competition